In [1]:
!pip install scikit-learn xgboost pandas numpy yfinance

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import yfinance as yf
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

df = yf.download("EURUSD=X", start="2005-01-01", end="2026-08-31")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Réinitialiser l'index pour récupérer la colonne de date
df = df.reset_index()
if 'Date' in df.columns:
    df = df.rename(columns={'Date': 'time'})

[*********************100%***********************]  1 of 1 completed


In [11]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df_clean = df.dropna()

features = ['Open', 'High', 'Low', 'Close']
X = df_clean[features]
y = df_clean['Target']

train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

In [12]:
from sklearn.ensemble import (
    RandomForestClassifier)

model = RandomForestClassifier(
    n_estimators=300,      # nb d'arbres
    max_depth=6,
    max_features='sqrt',   # aléa/split
    oob_score=True,
    random_state=42, n_jobs=-1)

model.fit(X_train, y_train)
print("OOB:", model.oob_score_)

OOB: 0.6067215668818161


In [13]:
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.62      0.83      0.71       578
           1       0.71      0.46      0.56       546

    accuracy                           0.65      1124
   macro avg       0.67      0.64      0.63      1124
weighted avg       0.66      0.65      0.64      1124

